<a href="https://colab.research.google.com/github/carlosdouglas1313/Livro-receitas/blob/master/Untitled14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd

# Step 1: Load the data
df_validadas = pd.read_csv('/content/horas_validadas_092025.csv')
df_planejadas = pd.read_excel('/content/horas_planejadas_092025.xlsx')

# Step 2: Process horas_validadas
df_validadas = df_validadas['Ano;Mês;Dia;Dia da Semana;IS;Destino;Atividade;Descrição;Horas;Tipo Horas;Status'].str.split(';', expand=True)
df_validadas.columns = ['Ano', 'Mês', 'Dia', 'Dia da Semana', 'IS', 'Destino', 'Atividade', 'Descrição', 'Horas', 'Tipo Horas', 'Status']
df_validadas['Horas'] = df_validadas['Horas'].astype(str)
df_validadas['Horas'] = df_validadas['Horas'].str.replace('.', ',', regex=False)
# Convert date columns to datetime in df_validadas
df_validadas = df_validadas.rename(columns={'Ano': 'year', 'Mês': 'month', 'Dia': 'day'})
df_validadas['Data'] = pd.to_datetime(df_validadas[['year', 'month', 'day']])
# Convert 'Horas' to numeric
df_validadas['Horas'] = df_validadas['Horas'].str.replace(',', '.', regex=False)
df_validadas['Horas'] = pd.to_numeric(df_validadas['Horas'], errors='coerce')


# Step 3: Process horas_planejadas
df_planejadas['IS'] = df_planejadas['IS - Nome Colaborador'].str.split(' ').str[0]
df_planejadas['Registrado'] = df_planejadas['Registrado'].astype(str)
df_planejadas['Registrado'] = df_planejadas['Registrado'].str.replace('.', ',', regex=False)
df_planejadas = df_planejadas[df_planejadas['Registrado'] != 'h']
# Ensure 'Data início' is in datetime format
df_planejadas['Data início'] = pd.to_datetime(df_planejadas['Data início'])
# Convert 'Registrado' to numeric
df_planejadas['Registrado'] = df_planejadas['Registrado'].str.replace(',', '.', regex=False)
df_planejadas['Registrado'] = pd.to_numeric(df_planejadas['Registrado'], errors='coerce')


# Step 4: Compare the dataframes including date
planejadas_tuples = set(tuple(row) for row in df_planejadas[['IS', 'Registrado', 'Data início']].values)
validadas_tuples = set(tuple(row) for row in df_validadas[['IS', 'Horas', 'Data']].values)

diff_tuples = planejadas_tuples - validadas_tuples

df_diff = pd.DataFrame(list(diff_tuples), columns=['IS', 'Registrado', 'Data início'])

# Step 5: Merge df_diff with df_planejadas to get additional columns and select/reorder
df_output = pd.merge(df_diff, df_planejadas[['IS', 'Registrado', 'Data início', 'IS - Nome Colaborador']],
                     on=['IS', 'Registrado', 'Data início'],
                     how='left')

# Convert 'Registrado' to numeric, replacing ',' with '.' first
df_output['Registrado'] = df_output['Registrado'].astype(str).str.replace(',', '.', regex=False)
df_output['Registrado'] = pd.to_numeric(df_output['Registrado'], errors='coerce')

df_output['Ano'] = df_output['Data início'].dt.year
df_output['Mês'] = df_output['Data início'].dt.month
df_output['Dia'] = df_output['Data início'].dt.day

df_output = df_output[['Ano', 'Mês', 'Dia', 'Data início', 'IS', 'IS - Nome Colaborador', 'Registrado']]

# Step 6: Save to Excel
output_month = df_output['Data início'].dt.month.iloc[0]
output_year = df_output['Data início'].dt.year.iloc[0]
output_filename = f"diferenca_planejado_{output_month:02d}{output_year}.xlsx"

df_output.to_excel(output_filename, index=False)

print(f"Resultado salvo em '{output_filename}'")

# Step 7: Display the differences
display(df_diff.head())
print(f"Number of differences found: {len(df_diff)}")

Resultado salvo em 'diferenca_planejado_092025.xlsx'


,IS,Registrado,Data início
0,MIDS,2.00,2025-09-11
1,MIDS,2.00,2025-09-10
2,MIDS,2.00,2025-09-23
3,LCOE1,3.67,2025-09-29
4,GLEO,2.67,2025-09-01


Number of differences found: 15
